In [2]:
'''Я продолжаю исследование оттока клиентов банка, которое проводила в рамках первой промежуточной аттестации.
В предыдущей работе я подготовила аналитическую выборку, изучила основные характеристики клиентов,
провела разведочный анализ данных и сформулировала гипотезы о возможных факторах оттока.
В данной работе я перехожу к построению моделей машинного обучения. Моя задача — обучить несколько моделей классификации,
сравнить их результаты и выбрать наиболее подходящую для прогнозирования оттока клиентов банка.

Также я проверю, насколько выбранная модель может определять клиентов с повышенным риском оттока,
и предложу рекомендации по использованию полученных результатов в банковской деятельности.

Дополнительно я рассмотрю возможность применения нейронных сетей для анализа текстовых обращений клиентов.

На первом этапе я подключаю необходимые библиотеки для работы с данными, построения графиков и последующего машинного обучения.
Затем загружаю датасет, который подготовила в первой аттестации. Повторно выполнять SQL-запросы
 и формировать VIEW мне не требуется, поскольку необходимые признаки уже рассчитаны.'''

'Я продолжаю исследование оттока клиентов банка, которое проводила в рамках первой промежуточной аттестации.\nВ предыдущей работе я подготовила аналитическую выборку, изучила основные характеристики клиентов,\nпровела разведочный анализ данных и сформулировала гипотезы о возможных факторах оттока.\nВ данной работе я перехожу к построению моделей машинного обучения. Моя задача — обучить несколько моделей классификации,\nсравнить их результаты и выбрать наиболее подходящую для прогнозирования оттока клиентов банка.\n\nТакже я проверю, насколько выбранная модель может определять клиентов с повышенным риском оттока,\nи предложу рекомендации по использованию полученных результатов в банковской деятельности.\n\nДополнительно я рассмотрю возможность применения нейронных сетей для анализа текстовых обращений клиентов.\n\nНа первом этапе я подключаю необходимые библиотеки для работы с данными, построения графиков и последующего машинного обучения.\nЗатем загружаю датасет, который подготовила в 

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Загрузка подготовленного датасета
from google.colab import files

uploaded = files.upload()

df = pd.read_csv("customer_churn_features.csv")

# Проверка загруженных данных
print("Размер датасета:", df.shape)

display(df.head())

Saving customer_churn_features.csv to customer_churn_features.csv
Размер датасета: (10000, 22)


,customer_id,age,gender,registration_date,annual_income,account_count,active_account_count,card_count,active_card_count,transaction_count,...,card_purchase_count,atm_withdrawal_count,transfer_out_count,utility_payment_count,transfer_count,loan_count,total_loan_amount,avg_interest_rate,last_transaction_date,churn
0,1,34,M,2025-07-12,455276.04,1,1,1,1,266,...,249,5,0,0,0,0,0.0,0.0,2026-01-27,0
1,2,54,M,2025-03-22,1945830.96,2,2,1,1,465,...,434,9,0,4,0,0,0.0,0.0,2026-01-26,0
2,3,18,F,2025-10-01,315794.16,1,1,1,1,155,...,144,2,1,1,1,0,0.0,0.0,2026-01-27,0
3,4,18,F,2025-11-22,647645.88,2,2,1,1,76,...,70,2,0,2,0,0,0.0,0.0,2026-01-25,0
4,5,49,M,2025-09-30,1061109.84,2,2,0,0,165,...,153,2,0,3,0,0,0.0,0.0,2026-01-27,0


In [4]:
# Общая информация о датасете
df.info()

# Проверка пропущенных значений
print("\nКоличество пропусков:")
display(df.isnull().sum())

# Проверка дубликатов
print("\nКоличество дубликатов:", df.duplicated().sum())

# Распределение целевого признака
print("\nРаспределение оттока клиентов:")
display(df["churn"].value_counts())

print("\nДоли классов, %:")
display((df["churn"].value_counts(normalize=True) * 100).round(2))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 22 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_id               10000 non-null  int64  
 1   age                       10000 non-null  int64  
 2   gender                    10000 non-null  object 
 3   registration_date         10000 non-null  object 
 4   annual_income             10000 non-null  float64
 5   account_count             10000 non-null  int64  
 6   active_account_count      10000 non-null  int64  
 7   card_count                10000 non-null  int64  
 8   active_card_count         10000 non-null  int64  
 9   transaction_count         10000 non-null  int64  
 10  total_transaction_amount  10000 non-null  float64
 11  avg_transaction_amount    10000 non-null  float64
 12  card_purchase_count       10000 non-null  int64  
 13  atm_withdrawal_count      10000 non-null  int64  
 14  transfe

,0
customer_id,0
age,0
gender,0
registration_date,0
annual_income,0
account_count,0
active_account_count,0
card_count,0
active_card_count,0
transaction_count,0



Количество дубликатов: 0

Распределение оттока клиентов:


,count
churn,
0,9303
1,697



Доли классов, %:


,proportion
churn,
0,93.03
1,6.97


In [5]:
'''Проверка показала, что данные в хорошем состоянии: пропусков и дубликатов нет, все 10 000 записей заполнены.
Целевой признак churn распределен неравномерно: 93,03% клиентов не ушли из банка, а 6,97% прекратили обслуживание.
Это важно учитывать при обучении моделей: обычная доля правильных ответов может быть высокой даже у модели,
которая плохо находит клиентов с риском оттока. Поэтому дальше я буду оценивать не только accuracy, но и precision, recall и F1-score.

Я исключу customer_id, так как это технический идентификатор, а не характеристика клиента.
Также преобразую даты регистрации и последней транзакции в числовые признаки. Для этого использую
фиксированную дату среза данных — 28 января 2026 года.'''

'Проверка показала, что данные в хорошем состоянии: пропусков и дубликатов нет, все 10 000 записей заполнены.\nЦелевой признак churn распределен неравномерно: 93,03% клиентов не ушли из банка, а 6,97% прекратили обслуживание.\nЭто важно учитывать при обучении моделей: обычная доля правильных ответов может быть высокой даже у модели,\nкоторая плохо находит клиентов с риском оттока. Поэтому дальше я буду оценивать не только accuracy, но и precision, recall и F1-score.\n\nЯ исключу customer_id, так как это технический идентификатор, а не характеристика клиента.\nТакже преобразую даты регистрации и последней транзакции в числовые признаки. Для этого использую\nфиксированную дату среза данных — 28 января 2026 года.'

In [6]:
# Подготовка признаков для модели

# Преобразование даты регистрации
df["registration_date"] = pd.to_datetime(df["registration_date"])

# Стаж клиента в днях на дату формирования выборки
reference_date = pd.Timestamp("2026-01-28")
df["customer_tenure_days"] = (
    reference_date - df["registration_date"]
).dt.days

# Удаление технических и потенциально проблемных признаков
X = df.drop(columns=[
    "customer_id",
    "registration_date",
    "last_transaction_date",
    "churn"
])

# Целевая переменная
y = df["churn"]

print("Количество признаков:", X.shape[1])
print("Количество объектов:", X.shape[0])
print("\nСписок признаков:")
print(X.columns.tolist())

Количество признаков: 19
Количество объектов: 10000

Список признаков:
['age', 'gender', 'annual_income', 'account_count', 'active_account_count', 'card_count', 'active_card_count', 'transaction_count', 'total_transaction_amount', 'avg_transaction_amount', 'card_purchase_count', 'atm_withdrawal_count', 'transfer_out_count', 'utility_payment_count', 'transfer_count', 'loan_count', 'total_loan_amount', 'avg_interest_rate', 'customer_tenure_days']


In [7]:
'''Подготовка прошла успешно. Для обучения осталось 19 признаков и 10 000 наблюдений.
 Технический идентификатор и исходные даты исключены, вместо даты регистрации добавлен стаж клиента в днях.
 Дату последней транзакции я не использую, чтобы не опираться на признак, который может быть тесно связан с фактом оттока.

Теперь я разделю данные на обучающую, валидационную и тестовую выборки. Сохраню исходное
соотношение классов, чтобы в каждой части была сопоставимая доля ушедших клиентов.'''

'Подготовка прошла успешно. Для обучения осталось 19 признаков и 10 000 наблюдений.\n Технический идентификатор и исходные даты исключены, вместо даты регистрации добавлен стаж клиента в днях.\n Дату последней транзакции я не использую, чтобы не опираться на признак, который может быть тесно связан с фактом оттока.\n\nТеперь я разделю данные на обучающую, валидационную и тестовую выборки. Сохраню исходное\nсоотношение классов, чтобы в каждой части была сопоставимая доля ушедших клиентов.'

In [8]:
from sklearn.model_selection import train_test_split

# Сначала выделяю 20% данных для тестирования
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Из оставшихся данных выделяю валидационную выборку
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.25,
    random_state=42,
    stratify=y_train_val
)

print("Обучающая выборка:", X_train.shape)
print("Валидационная выборка:", X_val.shape)
print("Тестовая выборка:", X_test.shape)

print("\nДоля оттока в выборках:")
print("Обучающая:", round(y_train.mean() * 100, 2), "%")
print("Валидационная:", round(y_val.mean() * 100, 2), "%")
print("Тестовая:", round(y_test.mean() * 100, 2), "%")

Обучающая выборка: (6000, 19)
Валидационная выборка: (2000, 19)
Тестовая выборка: (2000, 19)

Доля оттока в выборках:
Обучающая: 6.98 %
Валидационная: 6.95 %
Тестовая: 6.95 %


In [9]:
'''Разделение прошло правильно: 60% данных (6000 записей) выделено для обучения, по 20% — для проверки и тестирования.
Доля ушедших клиентов во всех трех выборках практически одинаковая —
около 7%. Значит, баланс классов удалось сохранить.
Теперь я подготовлю обработку признаков: категориальный признак gender преобразую
в числовой формат, а числовые признаки стандартизирую. Это необходимо, в частности, для корректной работы логистической регрессии.'''

'Разделение прошло правильно: 60% данных (6000 записей) выделено для обучения, по 20% — для проверки и тестирования.\nДоля ушедших клиентов во всех трех выборках практически одинаковая —\nоколо 7%. Значит, баланс классов удалось сохранить.\nТеперь я подготовлю обработку признаков: категориальный признак gender преобразую\nв числовой формат, а числовые признаки стандартизирую. Это необходимо, в частности, для корректной работы логистической регрессии.'

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# Разделяю признаки на числовые и категориальные
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

# Создаю обработку признаков
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("Числовые признаки:", len(numeric_features))
print("Категориальные признаки:", categorical_features)

Числовые признаки: 18
Категориальные признаки: ['gender']


In [11]:
'''Подготовка признаков настроена корректно: выделено 18 числовых признаков и один категориальный — gender.
Для числовых данных будет применяться стандартизация, а пол клиента будет преобразован в числовые категории.
Теперь я обучу первую модель — логистическую регрессию. Она будет базовой моделью, с которой можно
сравнить результаты более сложных алгоритмов.'''

'Подготовка признаков настроена корректно: выделено 18 числовых признаков и один категориальный — gender.\nДля числовых данных будет применяться стандартизация, а пол клиента будет преобразован в числовые категории.\nТеперь я обучу первую модель — логистическую регрессию. Она будет базовой моделью, с которой можно\nсравнить результаты более сложных алгоритмов.'

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score

# Создаю и обучаю модель
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

# Проверяю модель на валидационной выборке
y_val_pred = logistic_model.predict(X_val)
y_val_proba = logistic_model.predict_proba(X_val)[:, 1]

print("Результаты логистической регрессии:")
print(classification_report(y_val, y_val_pred, zero_division=0))

print("ROC-AUC:", round(roc_auc_score(y_val, y_val_proba), 4))

Результаты логистической регрессии:
              precision    recall  f1-score   support

           0       0.92      0.51      0.66      1861
           1       0.06      0.45      0.11       139

    accuracy                           0.50      2000
   macro avg       0.49      0.48      0.38      2000
weighted avg       0.86      0.50      0.62      2000

ROC-AUC: 0.5032


In [13]:
'''Логистическая регрессия показала слабый результат. Для класса ушедших клиентов recall составляет 0,45,
то есть модель находит около 45% таких клиентов. Однако precision равен всего 0,06: среди отмеченных
моделью клиентов большинство на самом деле не ушли. Значение ROC-AUC — 0,5032 — практически совпадает со случайным прогнозом.
Таким образом, логистическая регрессия пока не подходит для надежного прогнозирования оттока. Я проверю,
сможет ли более сложная модель лучше выявлять клиентов с риском ухода.'''

'Логистическая регрессия показала слабый результат. Для класса ушедших клиентов recall составляет 0,45,\nто есть модель находит около 45% таких клиентов. Однако precision равен всего 0,06: среди отмеченных\nмоделью клиентов большинство на самом деле не ушли. Значение ROC-AUC — 0,5032 — практически совпадает со случайным прогнозом.\nТаким образом, логистическая регрессия пока не подходит для надежного прогнозирования оттока. Я проверю,\nсможет ли более сложная модель лучше выявлять клиентов с риском ухода.'

In [14]:
from sklearn.ensemble import RandomForestClassifier

# Создаю модель случайного леса
random_forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# Обучаю модель
random_forest_model.fit(X_train, y_train)

# Проверяю на валидационной выборке
y_val_pred_rf = random_forest_model.predict(X_val)
y_val_proba_rf = random_forest_model.predict_proba(X_val)[:, 1]

print("Результаты случайного леса:")
print(classification_report(y_val, y_val_pred_rf, zero_division=0))

print("ROC-AUC:", round(roc_auc_score(y_val, y_val_proba_rf), 4))

Результаты случайного леса:
              precision    recall  f1-score   support

           0       0.93      1.00      0.96      1861
           1       0.00      0.00      0.00       139

    accuracy                           0.93      2000
   macro avg       0.47      0.50      0.48      2000
weighted avg       0.87      0.93      0.90      2000

ROC-AUC: 0.4607


In [15]:
from sklearn.tree import DecisionTreeClassifier

# Создаю модель дерева решений
decision_tree_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        class_weight="balanced",
        max_depth=5,
        min_samples_leaf=20,
        random_state=42
    ))
])

# Обучаю модель
decision_tree_model.fit(X_train, y_train)

# Проверяю на валидационной выборке
y_val_pred_dt = decision_tree_model.predict(X_val)
y_val_proba_dt = decision_tree_model.predict_proba(X_val)[:, 1]

print("Результаты дерева решений:")
print(classification_report(y_val, y_val_pred_dt, zero_division=0))

print("ROC-AUC:", round(roc_auc_score(y_val, y_val_proba_dt), 4))

Результаты дерева решений:
              precision    recall  f1-score   support

           0       0.92      0.13      0.23      1861
           1       0.07      0.84      0.13       139

    accuracy                           0.18      2000
   macro avg       0.49      0.49      0.18      2000
weighted avg       0.86      0.18      0.23      2000

ROC-AUC: 0.499


In [16]:
'''Дерево решений показало recall 0,84 для ушедших клиентов - находит большую часть оттока.
Но precision составляет только 0,07, то есть модель часто ошибочно относит оставшихся клиентов к группе риска.
ROC-AUC равен 0,499, поэтому в целом качество разделения классов остается низким.
Сравнение трех моделей показывает, что ни одна пока не дает надежного прогноза.
Перед итоговым сравнением я проверю, можно ли улучшить результаты, изменив порог классификации'''

'Дерево решений показало recall 0,84 для ушедших клиентов - находит большую часть оттока. \nНо precision составляет только 0,07, то есть модель часто ошибочно относит оставшихся клиентов к группе риска. \nROC-AUC равен 0,499, поэтому в целом качество разделения классов остается низким.\nСравнение трех моделей показывает, что ни одна пока не дает надежного прогноза. \nПеред итоговым сравнением я проверю, можно ли улучшить результаты, изменив порог классификации'

In [17]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score
)

# Собираю прогнозы трех моделей
models_proba = {
    "Логистическая регрессия": y_val_proba,
    "Случайный лес": y_val_proba_rf,
    "Дерево решений": y_val_proba_dt
}

# Сравниваю качество при разных порогах
results = []

for model_name, probabilities in models_proba.items():
    for threshold in [0.1, 0.2, 0.3, 0.4, 0.5]:
        predictions = (probabilities >= threshold).astype(int)

        results.append({
            "Модель": model_name,
            "Порог": threshold,
            "Precision": precision_score(y_val, predictions, zero_division=0),
            "Recall": recall_score(y_val, predictions, zero_division=0),
            "F1-score": f1_score(y_val, predictions, zero_division=0),
            "PR-AUC": average_precision_score(y_val, probabilities)
        })

results_df = pd.DataFrame(results)

display(results_df.round(3))

,Модель,Порог,Precision,Recall,F1-score,PR-AUC
0,Логистическая регрессия,0.1,0.070,1.000,0.130,0.076
1,Логистическая регрессия,0.2,0.070,1.000,0.130,0.076
2,Логистическая регрессия,0.3,0.070,1.000,0.130,0.076
3,Логистическая регрессия,0.4,0.070,1.000,0.130,0.076
4,Логистическая регрессия,0.5,0.063,0.446,0.111,0.076
5,Случайный лес,0.1,0.062,0.194,0.095,0.065
6,Случайный лес,0.2,0.064,0.022,0.032,0.065
7,Случайный лес,0.3,0.000,0.000,0.000,0.065
8,Случайный лес,0.4,0.000,0.000,0.000,0.065
9,Случайный лес,0.5,0.000,0.000,0.000,0.065


In [18]:
'''По таблице видно, что подбор порога почти не улучшил результаты.
Логистическая регрессия и дерево решений находят много ушедших клиентов, но часто ошибаются.
PR-AUC всех моделей находится около 0,07 — это почти не отличается от доли оттока в выборке (6,95%).
 Значит, пока модели плохо отличают ушедших клиентов от остальных.

Проверю, какие признаки содержат больше информации о целевом признаке churn.'''

'По таблице видно, что подбор порога почти не улучшил результаты. \nЛогистическая регрессия и дерево решений находят много ушедших клиентов, но часто ошибаются. \nPR-AUC всех моделей находится около 0,07 — это почти не отличается от доли оттока в выборке (6,95%).\n Значит, пока модели плохо отличают ушедших клиентов от остальных.\n\nПроверю, какие признаки содержат больше информации о целевом признаке churn.'

In [19]:
from sklearn.feature_selection import mutual_info_classif

# Преобразую признаки для анализа
X_train_processed = preprocessor.fit_transform(X_train)

# Получаю названия преобразованных признаков
feature_names = preprocessor.get_feature_names_out()

# Оцениваю связь признаков с оттоком
mi_scores = mutual_info_classif(
    X_train_processed,
    y_train,
    random_state=42
)

# Формирую таблицу результатов
feature_importance = pd.DataFrame({
    "Признак": feature_names,
    "Важность": mi_scores
}).sort_values(by="Важность", ascending=False)

display(feature_importance.head(10).round(4))

,Признак,Важность
12,num__utility_payment_count,0.0065
14,num__loan_count,0.0042
1,num__annual_income,0.0031
18,cat__gender_F,0.0028
13,num__transfer_count,0.0024
6,num__transaction_count,0.0013
11,num__transfer_out_count,0.0011
7,num__total_transaction_amount,0.0009
8,num__avg_transaction_amount,0.0009
2,num__account_count,0.0007


In [20]:
'''Результаты показывают, что связь отдельных признаков с оттоком очень слабая.
Наибольшую важность получили количество коммунальных платежей, количество кредитов и годовой доход.
Но даже их значения небольшие, поэтому нельзя утверждать, что какой-то один фактор хорошо объясняет уход клиентов.

Для дальнейшей проверки я выберу логистическую регрессию: у нее оказался самый высокий PR-AUC среди трех моделей
— 0,076. Теперь проверю ее на тестовой выборке, которую мы до этого не использовали при обучении и подборе порога.'''

'Результаты показывают, что связь отдельных признаков с оттоком очень слабая. \nНаибольшую важность получили количество коммунальных платежей, количество кредитов и годовой доход. \nНо даже их значения небольшие, поэтому нельзя утверждать, что какой-то один фактор хорошо объясняет уход клиентов.\n\nДля дальнейшей проверки я выберу логистическую регрессию: у нее оказался самый высокий PR-AUC среди трех моделей \n— 0,076. Теперь проверю ее на тестовой выборке, которую мы до этого не использовали при обучении и подборе порога.'

In [21]:
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# Использую выбранную модель и порог 0.4
y_test_proba = logistic_model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= 0.4).astype(int)

print("Результаты на тестовой выборке:")
print(classification_report(y_test, y_test_pred, zero_division=0))

print("ROC-AUC:", round(roc_auc_score(y_test, y_test_proba), 4))
print("PR-AUC:", round(average_precision_score(y_test, y_test_proba), 4))

Результаты на тестовой выборке:
              precision    recall  f1-score   support

           0       1.00      0.00      0.00      1861
           1       0.07      1.00      0.13       139

    accuracy                           0.07      2000
   macro avg       0.53      0.50      0.07      2000
weighted avg       0.94      0.07      0.01      2000

ROC-AUC: 0.4939
PR-AUC: 0.0768


In [22]:
'''Результат на тестовой выборке подтвердил, что логистическая регрессия пока не подходит для прогнозирования оттока.
- Recall = 1,00: модель нашла всех ушедших клиентов.
- Precision = 0,07: среди всех клиентов, которых она отнесла к группе риска, только около 7% действительно ушли.
- ROC-AUC = 0,4939, PR-AUC = 0,0768 — качество прогнозирования близко к случайному уровню.
Получается, при пороге 0,4 модель фактически относит почти всех клиентов к группе риска.
Это не дает банку полезного инструмента для работы с оттоком. В итоговой работе лучше честно указать это ограничение, а не называть модель успешной.

посчитаю, сколько клиентов модель отнесла к группе риска и сколько из них действительно ушли.'''

'Результат на тестовой выборке подтвердил, что логистическая регрессия пока не подходит для прогнозирования оттока.\n- Recall = 1,00: модель нашла всех ушедших клиентов.\n- Precision = 0,07: среди всех клиентов, которых она отнесла к группе риска, только около 7% действительно ушли.\n- ROC-AUC = 0,4939, PR-AUC = 0,0768 — качество прогнозирования близко к случайному уровню.\nПолучается, при пороге 0,4 модель фактически относит почти всех клиентов к группе риска. \nЭто не дает банку полезного инструмента для работы с оттоком. В итоговой работе лучше честно указать это ограничение, а не называть модель успешной.\n\nпосчитаю, сколько клиентов модель отнесла к группе риска и сколько из них действительно ушли.'

In [23]:
from sklearn.metrics import confusion_matrix

# Матрица ошибок
cm = confusion_matrix(y_test, y_test_pred)

print("Матрица ошибок:")
print(cm)

print("\nКоличество клиентов в тестовой выборке:", len(y_test))
print("Отнесены моделью к группе риска:", y_test_pred.sum())
print("Действительно ушли из банка:", y_test.sum())
print("Правильно найдено ушедших:", cm[1, 1])
print("Ошибочно отнесены к группе риска:", cm[0, 1])

Матрица ошибок:
[[   2 1859]
 [   0  139]]

Количество клиентов в тестовой выборке: 2000
Отнесены моделью к группе риска: 1998
Действительно ушли из банка: 139
Правильно найдено ушедших: 139
Ошибочно отнесены к группе риска: 1859


In [24]:
'''В ходе работы я подготовила данные для машинного обучения и построила три модели классификации:
логистическую регрессию, случайный лес и дерево решений.

Для оценки моделей использовала показатели Precision, Recall, F1-score, ROC-AUC и PR-AUC.
Дополнительно я проверила влияние разных порогов классификации и провела итоговое тестирование на отдельной выборке.
По результатам исследования выяснилось, что все три модели показали низкое качество прогнозирования оттока.
На тестовой выборке логистическая регрессия обнаружила всех ушедших клиентов, однако ошибочно отнесла к группе риска 1859 клиентов,
которые фактически не ушли из банка. Значение ROC-AUC составило 0,4939, а PR-AUC — 0,0768.

Таким образом, построенные модели пока нельзя рекомендовать для практического прогнозирования оттока клиентов.
Одной из возможных причин низкого качества является недостаточная информативность имеющихся признаков и сильный дисбаланс классов.
Рекомендации для банка
1. Собирать дополнительные данные о поведении клиентов, активности счетов и причинах прекращения обслуживания.
2. Анализировать историю обращений и жалоб клиентов для выявления признаков неудовлетворенности.
3. Использовать методы работы с несбалансированными данными при обучении моделей.
4. Проверить другие алгоритмы машинного обучения и подобрать параметры моделей.
5. Повторно оценивать качество моделей на новых данных перед внедрением в банковскую деятельность.'''

'В ходе работы я подготовила данные для машинного обучения и построила три модели классификации: \nлогистическую регрессию, случайный лес и дерево решений.\n\nДля оценки моделей использовала показатели Precision, Recall, F1-score, ROC-AUC и PR-AUC. \nДополнительно я проверила влияние разных порогов классификации и провела итоговое тестирование на отдельной выборке.\nПо результатам исследования выяснилось, что все три модели показали низкое качество прогнозирования оттока. \nНа тестовой выборке логистическая регрессия обнаружила всех ушедших клиентов, однако ошибочно отнесла к группе риска 1859 клиентов, \nкоторые фактически не ушли из банка. Значение ROC-AUC составило 0,4939, а PR-AUC — 0,0768.\n\nТаким образом, построенные модели пока нельзя рекомендовать для практического прогнозирования оттока клиентов. \nОдной из возможных причин низкого качества является недостаточная информативность имеющихся признаков и сильный дисбаланс классов.\nРекомендации для банка\n1. Собирать дополнительн

In [25]:
from sklearn.model_selection import train_test_split

# Разделяю данные на обучающую выборку (60%)
# и общую тестовую выборку (40%)
X_train, X_test_all, y_train, y_test_all = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42,
    stratify=y
)

# Делим тестовую выборку на две равные части
# Первая часть — для оценки качества
# Вторая часть — для построения прогнозов
X_val, X_forecast, y_val, y_forecast = train_test_split(
    X_test_all, y_test_all,
    test_size=0.5,
    random_state=42,
    stratify=y_test_all
)

print("Обучающая выборка:", X_train.shape)
print("Выборка для оценки:", X_val.shape)
print("Выборка для прогнозов:", X_forecast.shape)

print("\nДоля оттока:")
print("Обучение:", round(y_train.mean() * 100, 2), "%")
print("Оценка:", round(y_val.mean() * 100, 2), "%")
print("Прогноз:", round(y_forecast.mean() * 100, 2), "%")

Обучающая выборка: (6000, 19)
Выборка для оценки: (2000, 19)
Выборка для прогнозов: (2000, 19)

Доля оттока:
Обучение: 6.97 %
Оценка: 7.0 %
Прогноз: 6.95 %


In [26]:
'''Теперь я заново обучу логистическую регрессию, случайный лес и дерево решений на обучающей выборке.
Качество проверю только на выборке для оценки, не затрагивая данные для прогнозов.'''

'Теперь я заново обучу логистическую регрессию, случайный лес и дерево решений на обучающей выборке. \nКачество проверю только на выборке для оценки, не затрагивая данные для прогнозов.'

In [27]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)
import pandas as pd

models = {
    "Логистическая регрессия": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        ))
    ]),
    "Случайный лес": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]),
    "Дерево решений": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(
            class_weight="balanced",
            max_depth=5,
            min_samples_leaf=20,
            random_state=42
        ))
    ])
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_val)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)

    results.append({
        "Модель": name,
        "Precision": precision_score(y_val, predictions, zero_division=0),
        "Recall": recall_score(y_val, predictions, zero_division=0),
        "F1": f1_score(y_val, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_val, probabilities),
        "PR-AUC": average_precision_score(y_val, probabilities)
    })

results_df = pd.DataFrame(results).round(3)
display(results_df)

,Модель,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Логистическая регрессия,0.069,0.479,0.121,0.502,0.072
1,Случайный лес,0.000,0.000,0.000,0.499,0.072
2,Дерево решений,0.084,0.236,0.124,0.518,0.074


In [28]:
'''Порог 0.5 не всегда подходит для задачи оттока, особенно когда ушедших клиентов всего около 7%.
Проверим несколько порогов для дерева решений и выберем тот, при котором F1 будет максимальным.'''

'Порог 0.5 не всегда подходит для задачи оттока, особенно когда ушедших клиентов всего около 7%. \nПроверим несколько порогов для дерева решений и выберем тот, при котором F1 будет максимальным.'

In [29]:
from sklearn.metrics import precision_score, recall_score, f1_score

best_model = models["Дерево решений"]

val_probabilities = best_model.predict_proba(X_val)[:, 1]

threshold_results = []

for threshold in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]:
    predictions = (val_probabilities >= threshold).astype(int)

    threshold_results.append({
        "Порог": threshold,
        "Precision": precision_score(y_val, predictions, zero_division=0),
        "Recall": recall_score(y_val, predictions, zero_division=0),
        "F1": f1_score(y_val, predictions, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_results).round(3)
display(threshold_df)

,Порог,Precision,Recall,F1
0,0.05,0.069,0.943,0.129
1,0.10,0.069,0.943,0.129
2,0.15,0.069,0.943,0.129
3,0.20,0.069,0.943,0.129
4,0.25,0.069,0.943,0.129
5,0.30,0.069,0.943,0.129
6,0.40,0.069,0.943,0.129
7,0.50,0.084,0.236,0.124


In [30]:
# Выбранный порог классификации
best_threshold = 0.30

# Получаю вероятность оттока для клиентов
forecast_probabilities = best_model.predict_proba(X_forecast)[:, 1]

# Определяю прогнозируемый отток
forecast_predictions = (
    forecast_probabilities >= best_threshold
).astype(int)

# Формирую таблицу результатов
forecast_results = df.loc[X_forecast.index, ["customer_id"]].copy()

forecast_results["probability_churn"] = forecast_probabilities
forecast_results["predicted_churn"] = forecast_predictions

# Сортирую клиентов по вероятности оттока
forecast_results = forecast_results.sort_values(
    by="probability_churn",
    ascending=False
)

print("Количество клиентов для прогноза:", len(forecast_results))
print("Прогнозируемый отток:", forecast_results["predicted_churn"].sum())
print(
    "Доля клиентов с высоким риском:",
    round(forecast_results["predicted_churn"].mean() * 100, 2),
    "%"
)

display(forecast_results.head(10))

Количество клиентов для прогноза: 2000
Прогнозируемый отток: 1889
Доля клиентов с высоким риском: 94.45 %


,customer_id,probability_churn,predicted_churn
7471,7472,0.808322,1
3950,3951,0.808322,1
1323,1324,0.808322,1
7698,7699,0.808322,1
4586,4587,0.808322,1
8932,8933,0.808322,1
1638,1639,0.808322,1
6126,6127,0.808322,1
8666,8667,0.808322,1
2867,2868,0.689988,1


In [ ]:
'''Прогноз получен, но есть важный момент: модель отнесла к высокому риску 94,45% клиентов (1889 из 2000).
Это слишком большая доля для практического применения. При этом на проверочной выборке модель тоже часто ошибалась.

Посмотрим, как распределяются клиенты по вероятности оттока. Это поможет описать результаты и рекомендации.'''

In [31]:
# Разделяю клиентов на группы по вероятности оттока
forecast_results["risk_group"] = pd.cut(
    forecast_results["probability_churn"],
    bins=[-0.01, 0.3, 0.5, 1.0],
    labels=["Низкий риск", "Средний риск", "Высокий риск"]
)

risk_summary = (
    forecast_results["risk_group"]
    .value_counts()
    .reindex(["Низкий риск", "Средний риск", "Высокий риск"])
    .fillna(0)
    .astype(int)
    .reset_index()
)

risk_summary.columns = ["Группа риска", "Количество клиентов"]
risk_summary["Доля, %"] = (
    risk_summary["Количество клиентов"] / len(forecast_results) * 100
).round(2)

display(risk_summary)

,Группа риска,Количество клиентов,"Доля, %"
0,Низкий риск,111,5.55
1,Средний риск,1502,75.10
2,Высокий риск,387,19.35


In [ ]:
'''В рамках данной работы я рассмотрела возможность применения нейронных сетей для прогнозирования оттока клиентов банка.
Одним из перспективных направлений является анализ текстовых обращений клиентов.
Например, нейронная сеть может изучать сообщения, поступающие в службу поддержки, и определять
эмоциональную окраску текста, выявлять жалобы и признаки неудовлетворенности обслуживанием.

Для решения такой задачи можно использовать предварительно обученную языковую модель RuBERT, адаптировав ее для классификации текстовых обращений.
В качестве целевых классов можно выделить:
- положительные обращения
- нейтральные обращения
- отрицательные обращения

Результаты анализа могут дополнить существующую модель прогнозирования оттока. Например,
большое количество негативных обращений может указывать на повышенный риск ухода клиента.

В используемом наборе данных отсутствуют текстовые обращения клиентов, поэтому обучить и проверить
нейронную сеть в рамках данной работы невозможно. Я предлагаю использовать такую модель в дальнейшем исследовании при наличии соответствующих данных.'''

In [ ]:
'''В ходе выполнения второй промежуточной аттестации я продолжила исследование оттока клиентов банка и перешла
к построению моделей машинного обучения.
На первом этапе я подготовила данные и разделила их на три выборки: обучающую (60%), для оценки качества моделей (20%) и для прогнозирования (20%).
Затем я обучила три модели классификации: логистическую регрессию, случайный лес и дерево решений.
По результатам сравнения наилучшие показатели среди рассмотренных моделей продемонстрировало дерево решений.
Его значение F1 составило 0,124, а ROC-AUC — 0,518.
После этого я подобрала порог классификации и использовала выбранную модель для прогнозирования оттока
на отдельной выборке из 2000 клиентов. По результатам прогноза я выделила группы клиентов с низким, средним и высоким риском оттока.
Полученные результаты показали, что качество моделей пока недостаточно высокое для практического применения.
Это может быть связано с особенностями имеющихся признаков и дисбалансом классов. Для улучшения результатов необходимо
расширить набор данных, добавить дополнительные характеристики поведения клиентов и проверить другие алгоритмы машинного обучения.
Также я рассмотрела возможность применения нейронных сетей для анализа текстовых обращений клиентов. Для реализации
этой задачи в дальнейшем можно использовать языковую модель RuBERT.
Таким образом, в ходе работы я получила практический опыт подготовки данных, обучения моделей классификации,
оценки их качества и формирования прогнозов. Поставленная учебная задача была выполнена.'''